# 📊 SET_STS_BUENO: Análisis de Calibration Sets Alfanuméricos

**Notebook para procesar sets completos con CalibSetNumber alfanuméricos.**

Este notebook usa la clase `SetSTS` para analizar:
- **RESIST_SET**: Calibraciones con resistencias de precisión (carpeta `resistences/`)
- **FRAME_SET**: Calibraciones de sensores de frame (carpeta `frame_sensors/`)

## 📋 Diferencias vs SET_BUENO.ipynb

| Característica | SET_BUENO | SET_STS_BUENO |
|---------------|-----------|---------------|
| CalibSetNumber | Numérico (3, 4, 5...) | Alfanumérico (RESIST_SET0, FRAME_SET1) |
| Sensores raised | Sí (dinámicos por set) | No |
| Sensores discarded | Sí (config YAML) | No |
| Referencia | Dinámica (raised sensors) | Fija (configurable) |
| Carpeta datos | `Data/Frame_Sensors/` | `frame_sensors/` o `resistences/` |
| IDs sensores | Numéricos | Alfanuméricos o numéricos |

## 🎯 Cómo usar

1. **Celda 1**: Setup de paths e imports
2. **Celda 2**: **CONFIGURAR AQUÍ** - Selecciona tipo de set y parámetros
3. **Celda 3**: Procesar sets (grouping, offsets, repeatability)
4. **Celdas 4-5**: Plots de comparación global

## ⚙️ Parámetros configurables (Celda 2)

- `CALIBSET_PATTERN = 'RESIST_SET'` → Filtra por patrón (o `'FRAME_SET'`)
- `selected_sets = ['RESIST_SET0', 'RESIST_SET1']` → Sets específicos a procesar
- `ref_channel = 2` → Canal de referencia fijo (1-14)
- `data_folder = 'resistences'` → Carpeta de datos (`'resistences'` o `'frame_sensors'`)
- `tmin, tmax` → Ventana temporal para offsets (minutos)

---

In [1]:
# ═══════════════════════════════════════════════════════════════
# 📦 SETUP - Paths e Imports
# ═══════════════════════════════════════════════════════════════

import sys
from pathlib import Path
import pandas as pd
import importlib

# Configurar repo root
repo_root = Path('..').resolve().parent if (Path('..').resolve().name == 'RTD_Calibration_VGP') else Path('..').resolve()

if (repo_root / 'RTD_Calibration_VGP').exists():
    sys.path.insert(0, str(repo_root))
    print(f'✅ Repo root: {repo_root}')
else:
    sys.path.insert(0, str(Path('..').resolve()))
    print(f'⚠️  Fallback path: {Path("..").resolve()}')

# Reload modules para usar última versión
import RTD_Calibration_VGP.src.runSTS as runsts_mod
import RTD_Calibration_VGP.src.setSTS as setsts_mod

print(f'🔄 Reloading runSTS from: {getattr(runsts_mod, "__file__", "unknown")}')
importlib.reload(runsts_mod)

print(f'🔄 Reloading setSTS from: {getattr(setsts_mod, "__file__", "unknown")}')
importlib.reload(setsts_mod)

# Imports finales
from RTD_Calibration_VGP.src.logfile import Logfile
from RTD_Calibration_VGP.src.setSTS import SetSTS

print('✅ Modules imported successfully (runSTS + setSTS reloaded)')

✅ Repo root: /Users/vicky/Desktop/rtd-calibration-ana
🔄 Reloading runSTS from: /Users/vicky/Desktop/rtd-calibration-ana/RTD_Calibration_VGP/src/runSTS.py
🔄 Reloading setSTS from: /Users/vicky/Desktop/rtd-calibration-ana/RTD_Calibration_VGP/src/setSTS.py
✅ Modules imported successfully (runSTS + setSTS reloaded)


In [2]:
# ═══════════════════════════════════════════════════════════════
# ⚙️  CONFIGURACIÓN - MODIFICAR AQUÍ
# ═══════════════════════════════════════════════════════════════

# 1️⃣  Tipo de set a analizar (RESIST_SET o FRAME_SET)
CALIBSET_PATTERN = 'RESIST_SET'  # Cambiar a 'RESIST_SET' si es necesario

# 2️⃣  Carpeta de datos (debe coincidir con el tipo de set)
#     'resistences' → RESIST_SET
#     'frame_sensors' → FRAME_SET
data_folder = 'Resistences'  # Cambiar a 'Resistences' si es necesario

# 3️⃣  Sets específicos a procesar (None = todos los que coincidan con el patrón)
selected_sets = ['RESIST_SET5']  # Ejemplo: ['FRAME_SET0', 'FRAME_SET1']
# selected_sets = None  # Descomentar para procesar TODOS

# 4️⃣  Canal de referencia fijo (1-14)
ref_channel = 2  # Cambiar si necesitas otra referencia

# 5️⃣  Ventana temporal para cálculo de offsets (minutos desde inicio del run)
tini = 1  # Minuto inicial
tend = 20  # Minuto final

# 6️⃣  Directorio de salida para plots y CSVs
save_dir = f'outputs/offset_repeatability_{CALIBSET_PATTERN}'

# 7️⃣  Opciones de guardado
write_csv = True    # Guardar CSVs de estadísticas
write_excel = False  # Guardar Excel (opcional)

# ═══════════════════════════════════════════════════════════════

print("📋 Configuration Summary:")
print(f"  Pattern: {CALIBSET_PATTERN}")
print(f"  Data folder: {data_folder}")
print(f"  Selected sets: {selected_sets}")
print(f"  Reference channel: {ref_channel}")
print(f"  Time window: {tini}-{tend} min")
print(f"  Output directory: {save_dir}")

📋 Configuration Summary:
  Pattern: RESIST_SET
  Data folder: Resistences
  Selected sets: ['RESIST_SET5']
  Reference channel: 2
  Time window: 1-20 min
  Output directory: outputs/offset_repeatability_RESIST_SET


In [3]:
# ═══════════════════════════════════════════════════════════════
# 🔄 PROCESAMIENTO - Cargar LogFile y Procesar Sets
# ═══════════════════════════════════════════════════════════════

# Cargar LogFile
logfile_path = (repo_root / 'RTD_Calibration_VGP' / 'data' / 'LogFile.csv').resolve()

if not logfile_path.exists():
    raise FileNotFoundError(f"❌ LogFile not found: {logfile_path}")

print(f"📂 Loading LogFile from: {logfile_path}")
logfiles = Logfile(filepath=str(logfile_path))
logfile_df = logfiles.log_file

print(f"✅ LogFile loaded: {len(logfile_df)} rows\n")

# Crear instancia de SetSTS
print(f"🔧 Creating SetSTS instance (data_folder='{data_folder}')...")
set_instance = SetSTS(logfile=logfile_df, data_folder=data_folder)

# 1. Agrupar runs por CalibSetNumber
print(f"\n{'='*60}")
print("1️⃣  GROUPING RUNS BY SET")
print(f"{'='*60}")
set_instance.group_runs_by_set(calibset_pattern=CALIBSET_PATTERN, selected_sets=selected_sets)

# 2. Calcular offsets y RMS
print(f"\n{'='*60}")
print("2️⃣  CALCULATING OFFSETS AND RMS")
print(f"{'='*60}")
set_instance.calculate_offsets_and_rms(selected_sets=selected_sets, tini=tini, tend=tend)

# 3. Calcular repetibilidad de offsets
print(f"\n{'='*60}")
print("3️⃣  OFFSET REPEATABILITY ANALYSIS")
print(f"{'='*60}")
set_instance.offset_repeatability(
    tini=tini,
    tend=tend,
    save_dir=save_dir,
    selected_sets=selected_sets,
    ref_channel=ref_channel,
    write_csv=write_csv,
    write_excel=write_excel
)

print(f"\n{'='*60}")
print("✅ SET PROCESSING COMPLETE")
print(f"{'='*60}")
print(f"📊 Results saved in: {save_dir}")
print(f"📈 You can now run the plotting cells below")

📂 Loading LogFile from: /Users/vicky/Desktop/rtd-calibration-ana/RTD_Calibration_VGP/data/LogFile.csv
CSV file loaded successfully from '/Users/vicky/Desktop/rtd-calibration-ana/RTD_Calibration_VGP/data/LogFile.csv'.
✅ LogFile loaded: 832 rows

🔧 Creating SetSTS instance (data_folder='Resistences')...

1️⃣  GROUPING RUNS BY SET

🔄 Processing CalibSetNumber: RESIST_SET5
Archivo de temperatura encontrado: /Users/vicky/Desktop/rtd-calibration-ana/RTD_Calibration_VGP/data/temperature_files/Resistences/RESIST_SET5/20251110_air_STSr10_STSr9_PDHD-HP-49-PDHD-HP-60_1.txt
Valores NaN (contador): 0
Empty DataFrame
Columns: [datetime, channel, value]
Index: []
Archivo de temperatura procesado correctamente: /Users/vicky/Desktop/rtd-calibration-ana/RTD_Calibration_VGP/data/temperature_files/Resistences/RESIST_SET5/20251110_air_STSr10_STSr9_PDHD-HP-49-PDHD-HP-60_1.txt
No se detectaron canales defectuosos.
Valores de sensores extraídos (antes de filtrado y conversión): ['PDHD-HP-13' 'PDHD-HP-19' 'PDH

In [4]:
# ═══════════════════════════════════════════════════════════════
# 🔍 DEBUG - Inspeccionar global_stats
# ═══════════════════════════════════════════════════════════════

print("📊 Global stats collected:")
print(f"  Sets processed: {list(set_instance.global_stats.keys())}")

for set_num, stats in set_instance.global_stats.items():
    print(f"\n  Set: {set_num}")
    print(f"    Means: {stats.get('means', {})}")
    print(f"    Sigmas: {stats.get('sigmas', {})}")
    
if not set_instance.global_stats:
    print("⚠️  WARNING: global_stats is EMPTY!")
    print("\nChecking runs_by_set...")
    for set_num, runs in set_instance.runs_by_set.items():
        print(f"\n  Set {set_num}: {len(runs)} runs")
        for fname, run_obj in runs.items():
            print(f"    - {fname}")
            if hasattr(run_obj, 'sensor_mapping'):
                print(f"      Sensors: {list(run_obj.sensor_mapping.values())}")
            else:
                print(f"      ⚠️  No sensor_mapping!")

📊 Global stats collected:
  Sets processed: ['RESIST_SET5']

  Set: RESIST_SET5
    Means: {'PDHD-HP-13': np.float64(-31.694381254867736), 'PDHD-HP-21': np.float64(-17.78620869840355), 'PDHD-HP-28': np.float64(-1.3260333674067681), 'PDHD-HP-30': np.float64(-47.675461034852134), 'PDHD-HP-31': np.float64(-30.043535314447126), 'PDHD-HP-39': np.float64(-21.26114812110617), 'PDHD-HP-44': np.float64(-41.388709939641736), 'PDHD-HP-45': np.float64(-13.864083479361513), 'PDHD-HP-52': np.float64(-31.93909898267135), 'PDHD-HP-55': np.float64(-13.509609107281985), 'PDHD-HP-56': np.float64(-39.97065778329472), '1010': np.float64(219844.7488742455), '1009': np.float64(219885.83260321745)}
    Sigmas: {'PDHD-HP-13': np.float64(0.03746710877374498), 'PDHD-HP-21': np.float64(0.02169538502826586), 'PDHD-HP-28': np.float64(0.02058777704476954), 'PDHD-HP-30': np.float64(0.034737849953978536), 'PDHD-HP-31': np.float64(0.022291670832812634), 'PDHD-HP-39': np.float64(0.07006686787113495), 'PDHD-HP-44': np.fl

In [5]:
# ═══════════════════════════════════════════════════════════════
# 📊 PLOT - Global Mean Offsets
# ═══════════════════════════════════════════════════════════════

print("📊 Plotting global mean offsets...\n")
set_instance.plot_global_means(selected_sets=selected_sets, save_dir=save_dir)
print("\n✅ Global means plot generated")

📊 Plotting global mean offsets...

💾 Global means plot saved: outputs/offset_repeatability_RESIST_SET/global_means_comparison.png

✅ Global means plot generated


In [6]:
# ═══════════════════════════════════════════════════════════════
# 📊 PLOT - Global Sigmas (Repeatability)
# ═══════════════════════════════════════════════════════════════

print("📊 Plotting global sigmas (repeatability)...\n")
set_instance.plot_global_sigmas(selected_sets=selected_sets, save_dir=save_dir)
print("\n✅ Global sigmas plot generated")

📊 Plotting global sigmas (repeatability)...

💾 Global sigmas plot saved: outputs/offset_repeatability_RESIST_SET/global_sigmas_comparison.png

✅ Global sigmas plot generated


---

## 📌 Resultados Generados

El procesamiento habrá creado los siguientes archivos en `outputs/offset_repeatability_[PATTERN]/`:

### 🖼️ Plots individuales por set:
- `[CalibSetNumber]_offset_repeatability.png` - Gráfico de repetibilidad con 14 subplots

### 📊 Plots de comparación global:
- `global_means_comparison.png` - Comparación de medias entre sets
- `global_sigmas_comparison.png` - Comparación de repetibilidad entre sets

### 📄 CSVs de estadísticas (si `write_csv=True`):
- `[CalibSetNumber]_global_means.csv` - Medias de offset por sensor
- `[CalibSetNumber]_global_sigmas.csv` - Sigmas (repetibilidad) por sensor
- `skipped_runs.csv` - Runs excluidos del análisis

---

## 🔧 Troubleshooting

**Problema**: No se encuentran archivos de temperatura
- **Solución**: Verifica que `data_folder` coincida con el tipo de set:
  - `RESIST_SET` → `data_folder = 'resistences'`
  - `FRAME_SET` → `data_folder = 'frame_sensors'`

**Problema**: Sensores no encontrados en el mapping
- **Solución**: Verifica que el LogFile tenga las columnas S1-S20 correctamente pobladas para ese CalibSetNumber

**Problema**: Canal de referencia no válido
- **Solución**: Cambia `ref_channel` a un canal que exista en tus datos (típicamente 1-14)

---